In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification


In [2]:
# download measurements
%run ../src/get_sdss_data.py
# match morphology labels
%run ../src/join_morph_sdss.py
# impute missing values, engineers color and brightness features, removes outliers, and writes cleaned parquet outputs
%run ../src/clean_data.py

SDSS data has already been downloaded (using downloaded data)...
Loading morphology data from file...
Converting de Vaucouleurs coordinates for matching


100%|█████████████████████████████████████████████████████████████████████████| 22999/22999 [00:00<00:00, 27118.30it/s]


Converting SDSS coordinates for matching


100%|███████████████████████████████████████████████████████████████████████| 605174/605174 [00:19<00:00, 30761.46it/s]

 34%|███████████████████████▉                                               | 204068/605174 [00:27<00:14, 27882.73it/s]


 69%|████████████████████████████████████████████████▊                      | 415684/605174 [00:35<00:05, 32536.37it/s]


100%|███████████████████████████████████████████████████████████████████████| 605174/605174 [00:41<00:00, 14504.09it/s]


Matching SDSS data to de Vaucouleurs data...


TypeError: concatenate() got an unexpected keyword argument 'dtype'

FileNotFoundError: [Errno 2] No such file or directory: '../data/data_full.parquet'

## import data and set some constants

In [3]:
data = pd.read_parquet('../data/data_full.parquet')
print(data.columns)
FLOAT_COLS = [
            'T',
            'ra',
            'dec',
            'u',
            'g',
            'r',
            'i',
            'z',
            'petroMag_r',
            'petroR50_r',
            'petroR90_r'
        ]
COLS_TO_DROP = [  # these columns are not physical, so they obviously will not relate to morphology
            '_RAJ2000',
            '_DEJ2000',
            'sidx',
            'petroR50_r',
            'petroR90_r',
            'petroMag_r',
            'u',
            'g',
            'r',
            'i',
            'z'
        ]

FileNotFoundError: [Errno 2] No such file or directory: '../data/data_full.parquet'

## convert columns to float

In [ ]:
# cast some columns as float
for col in FLOAT_COLS:
    data[col] = pd.to_numeric(data[col], errors="coerce")

## data imputation

In [ ]:
COLS_TO_IMPUTE = ['T', 'u', 'g', 'r', 'i', 'z', 'petroMag_r', 'petroR50_r', 'petroR90_r']

data['imputed_brightness'] = data['z'].isna().astype(int)
imputer = KNNImputer(n_neighbors=50, weights='uniform')
data[COLS_TO_IMPUTE] = imputer.fit_transform(data[COLS_TO_IMPUTE])

## derive some features

In [ ]:
# calculate surface brightness
petroMag_r_half = data['petroMag_r'] + 0.7562
half_area_r = np.pi * data['petroR50_r']**2
data['sb50_r'] = petroMag_r_half + 2.512*np.log10(half_area_r)

# calculate surface brightness concentration
data['sb_conc_r'] = data['petroR90_r']/data['petroR50_r']

# calculate colors
data['ug_color'] = data['u'] - data['g']
data['ur_color'] = data['u'] - data['r']
data['ui_color'] = data['u'] - data['i']
data['uz_color'] = data['u'] - data['z']
data['gr_color'] = data['g'] - data['r']
data['gi_color'] = data['g'] - data['i']
data['gz_color'] = data['g'] - data['z']
data['ri_color'] = data['r'] - data['i']
data['rz_color'] = data['r'] - data['z']
data['iz_color'] = data['i'] - data['z']

# drop needless columns
data = data.drop(columns=COLS_TO_DROP)

## remove outliers

In [ ]:
OUT_COLS = ['T', 'ug_color', 'ur_color', 'ui_color', 'uz_color', 'gr_color', 'gi_color', 'gz_color', 'ri_color', 'rz_color', 'iz_color', 'sb50_r', 'sb_conc_r']

for cc in OUT_COLS:
    
    nonan = np.array(data[cc])[~np.isnan(np.array(data[cc]))]
    plt.boxplot(nonan, vert=False)
    plt.title(cc+' before outlier removal')
    plt.show()

    fq = np.quantile(nonan, 0.25)
    tq = np.quantile(nonan, 0.75)
    iqr = tq - fq
    ubound = tq + 1.5*iqr
    lbound = fq - 1.5*iqr

    data = data[ data[cc]<ubound ]
    data = data[ data[cc]>lbound ]

    nonan = np.array(data[cc])[~np.isnan(np.array(data[cc]))]
    plt.boxplot(nonan, vert=False)
    plt.title(cc+' after outlier removal')
    plt.show()

## drop a bunch of stuff

In [ ]:
# data.dropna(inplace=True)

## visualize

In [ ]:
x = data['ur_color'].to_numpy()
y = data['T'].to_numpy()

xy = np.vstack([x,y])
z = gaussian_kde(xy)(xy)

idx = z.argsort()
x, y, z = x[idx], y[idx], z[idx]

fig, ax = plt.subplots()
ax.scatter(x, y, c=z, s=50)
# ax.set_xlim([-5.,20.])
ax.set_ylim([-10.,20.])
plt.show()

In [ ]:
data

# Make Random Forest

In [ ]:
# train test split
RFXTrain, RFXTest, RFYTrain, RFYTest = train_test_split(x, y, random_state=42)
# forest of random
RandomForest = RandomForestClassifier(n_estimators=100, random_state=42)
# fit to model via training data use: RandomForest.score(RFXTest, RFYTest) to get accuracy
RandomForest.fit(RFXTrain, RFYTrain)